# 01 — Exploratory Data Analysis

Focus: understand class imbalance, feature-outcome correlations within the *control arm*
(uncontaminated by treatment), and mutual information as a proxy for heterogeneity signal.

In [ ]:
import sys; sys.path.insert(0, '..')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns
from sklearn.feature_selection import mutual_info_classif
from src.data import load_data, FEATURE_COLS

plt.rcParams.update({'figure.dpi': 130, 'font.size': 11})

In [ ]:
df = load_data(percent10=True)
treated = df[df['treatment'] == 1]
control = df[df['treatment'] == 0]
print(f'Dataset: {len(df):,} rows  |  Treated: {len(treated):,}  |  Control: {len(control):,}')

## 1. Label base rates and class imbalance

In [ ]:
labels = [l for l in ['visit', 'conversion'] if l in df.columns]
for lbl in labels:
    for arm_name, arm_df in [('Overall', df), ('Treated', treated), ('Control', control)]:
        rate = arm_df[lbl].mean()
        print(f'{lbl:12s} | {arm_name:8s}: {rate:.4%}')
    print()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
for ax, lbl in zip(axes, labels):
    rates = {
        'Treated': treated[lbl].mean(),
        'Control': control[lbl].mean()
    }
    bars = ax.bar(rates.keys(), [v * 100 for v in rates.values()],
                  color=['#1f77b4', '#ff7f0e'])
    ax.yaxis.set_major_formatter(mtick.PercentFormatter())
    ax.set_title(f'{lbl} rate by arm')
    ax.set_ylabel('Rate (%)')
    for bar, v in zip(bars, rates.values()):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
                f'{v:.3%}', ha='center', va='bottom', fontsize=9)
plt.tight_layout()
plt.savefig('../reports/label_rates.png', bbox_inches='tight')
plt.show()

## 2. Feature distributions (projected, so interpret cautiously)

In [ ]:
fig, axes = plt.subplots(3, 4, figsize=(14, 10))
axes = axes.flatten()
for i, col in enumerate(FEATURE_COLS):
    axes[i].hist(treated[col].values, bins=40, alpha=0.5, label='Treated', density=True, color='#1f77b4')
    axes[i].hist(control[col].values, bins=40, alpha=0.5, label='Control', density=True, color='#ff7f0e')
    axes[i].set_title(col)
    axes[i].set_yticks([])
axes[0].legend()
fig.suptitle('Feature distributions by arm  (randomly projected — do not over-interpret shapes)', y=1.01)
plt.tight_layout()
plt.savefig('../reports/feature_dists.png', bbox_inches='tight')
plt.show()

## 3. Correlation with `visit` within the control arm

Control arm is uncontaminated by treatment — these correlations are our best signal for
which features carry baseline propensity and are best candidates for CUPED adjustment.

In [ ]:
ctrl_corr = control[FEATURE_COLS + ['visit']].corr()['visit'].drop('visit').sort_values(key=abs, ascending=False)

fig, ax = plt.subplots(figsize=(7, 4))
colors = ['#d62728' if v < 0 else '#1f77b4' for v in ctrl_corr]
ax.barh(ctrl_corr.index, ctrl_corr.values, color=colors)
ax.axvline(0, color='black', linewidth=0.8)
ax.set_xlabel('Pearson correlation with visit (control arm)')
ax.set_title('Feature–outcome correlation (control only, uncontaminated by treatment)')
plt.tight_layout()
plt.savefig('../reports/feature_visit_corr.png', bbox_inches='tight')
plt.show()
print(ctrl_corr.round(4))

## 4. Mutual information with `visit` (treatment + control combined)

In [ ]:
X = df[FEATURE_COLS].values
y = df['visit'].values
mi = mutual_info_classif(X, y.astype(int), random_state=42)
mi_series = pd.Series(mi, index=FEATURE_COLS).sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(7, 4))
ax.bar(mi_series.index, mi_series.values, color='#2ca02c')
ax.set_ylabel('Mutual information with visit')
ax.set_title('Feature heterogeneity signal (mutual information with outcome)')
plt.tight_layout()
plt.savefig('../reports/mutual_info.png', bbox_inches='tight')
plt.show()
print('Top features by MI with visit:')
print(mi_series.round(5))

## 5. Treatment imbalance summary

This imbalance (~85/15) is **the main design challenge** — it directly motivates choosing the X-learner over the T-learner.

In [ ]:
fig, ax = plt.subplots(figsize=(4, 4))
sizes = [len(treated), len(control)]
labels_pie = [f'Treated\n{len(treated)/len(df):.1%}', f'Control\n{len(control)/len(df):.1%}']
ax.pie(sizes, labels=labels_pie, colors=['#1f77b4', '#ff7f0e'],
       startangle=90, autopct='%1.1f%%', textprops={'fontsize': 11})
ax.set_title('Treatment imbalance\n(motivates X-learner over T-learner)')
plt.tight_layout()
plt.savefig('../reports/treatment_imbalance.png', bbox_inches='tight')
plt.show()